In [ ]:
import os
import torch

# load all environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

# Generate and save Keys

In [ ]:
from save_outputs import main as generate_and_save_outputs

%load_ext autoreload
%autoreload 2

In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
dataset = "example_dataset"
num_samples = 10
save_path = "model_outputs.pt"

generate_and_save_outputs(
    model_name,
    dataset,
    save_path,
    micro_bs=num_samples,
    seq_len=2048,
)

# Load Keys

In [ ]:
BASE_DIR = os.getenv("BASE_DIR", ".")
save_path = os.path.join(BASE_DIR, "model_outputs.pt")

tensors = torch.load(save_path, weights_only=False)

print(tensors.keys())

In [ ]:
pkv = tensors['past_key_values']  # Expect a DynamicCache object

# retrieve all keys into a single tensor of shape (B, T, L, H, head_dim) - B=batch size, T=sequence length, L=number of layers, H=number of heads, head_dim=head dimension
all_keys = []
for layer_idx, layer_cache in enumerate(pkv.layers):
    all_keys.append(layer_cache.keys)

keys = torch.stack(all_keys, dim=2)  # shape (B, T, L, H, head_dim)
print("All keys tensor shape:", keys.shape)

# Metrics

In [ ]:
def mse(a, b):
    return ((a - b) ** 2).mean()

def abs_error(a, b):
    return (a - b).abs().sum(dim=(-1,-2)).mean()

def relative_error(a, b):
    return ((a - b).abs().sum() / a.abs().sum().clamp(min=1e-12))

def compute_svd_error(A, U, S, V, error_fn):
    # Reconstruct A from U, S, V
    A_reconstructed = (U * S.unsqueeze(-2)) @ V
    return error_fn(A, A_reconstructed)

def compression_ratio(A, U, S, V):
    original_size = A.numel()
    compressed_size = U.numel() + S.numel() + V.numel()
    return original_size / compressed_size

def print_all_errors(A, U, S, V):
    print("MSE:", compute_svd_error(A, U, S, V, mse).item())
    print("Absolute Error:", compute_svd_error(A, U, S, V, abs_error).item())
    print("Relative Error:", compute_svd_error(A, U, S, V, relative_error).item())
    print("Compression Ratio:", compression_ratio(A, U, S, V))

# SVD

### Truncated SVD

In [ ]:
from utils import full_svd, truncated_svd

%load_ext autoreload
%autoreload 2

In [ ]:
full_svd_keys = full_svd(keys)
U, S, V = full_svd_keys
print("keys.shape:", keys.shape)
print("U.shape:", U.shape)
print("S.shape:", S.shape)
print("V.shape:", V.shape)

In [ ]:
trunc_svd_keys = truncated_svd(keys, 4)
U, S, V = trunc_svd_keys
print("keys.shape:", keys.shape)
print("U.shape:", U.shape)
print("S.shape:", S.shape)
print("V.shape:", V.shape)

In [ ]:
full_svd_keys = full_svd(keys)
print(" ===== Errors for full SVD:")
print_all_errors(keys, *full_svd_keys)

In [ ]:
import matplotlib.pyplot as plt

truncated_svd_keys = {}
ranks = list(range(1, keys.shape[-1] + 1))
plt.figure(figsize=(10, 6))
for rank in ranks:
    truncated_svd_keys[rank] = {'trunc_keys': truncated_svd(keys, rank)}
    truncated_svd_keys[rank]['relative_err'] = compute_svd_error(keys, *(truncated_svd_keys[rank]['trunc_keys']), relative_error).item()
    truncated_svd_keys[rank]['compression_ratio'] = compression_ratio(keys, *(truncated_svd_keys[rank]['trunc_keys']))

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(ranks, [truncated_svd_keys[rank]['relative_err'] for rank in ranks], 'g-', label='Relative Error')
ax2.plot(ranks, [truncated_svd_keys[rank]['compression_ratio'] for rank in ranks], 'b-', label='Compression Ratio')
ax1.set_xlabel('Rank')
ax1.set_ylabel('Relative Error', color='g')
ax2.set_ylabel('Compression Ratio', color='b')
plt.title('Relative Error and Compression Ratio vs Rank')
plt.show()
